# Wikimedia XML/BZ2 to chunked JSON

Run this notebook in Google Colab from the project root, or upload the repository folder so `wikiparser/dump_converter.py` is available. The converter accepts either a single `.xml.bz2` dump URL/path or a Wikimedia directory URL ending with `/`.

In [ ]:
%pip install -q mwparserfromhell requests

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
module_path = project_root / "wikiparser" / "dump_converter.py"

if not module_path.exists():
    candidates = list(Path("/content").rglob("wikiparser/dump_converter.py"))
    if candidates:
        project_root = candidates[0].parents[1]
        module_path = candidates[0]

if not module_path.exists():
    raise FileNotFoundError(
        "Could not find wikiparser/dump_converter.py. Upload this repository to Colab, "
        "then run the notebook from the repository root."
    )

sys.path.insert(0, str(project_root))
print(f"Using converter from: {module_path}")

## Settings

`SOURCE` can be a catalog URL, a single dump URL, or a local file path. Use small `LIMIT` and `ARCHIVE_LIMIT` values for a smoke test before a full run.

In [ ]:
SOURCE = "https://dumps.wikimedia.org/other/mediawiki_content_current/enwiki/2026-05-01/xml/bzip2/"
OUTPUT_DIR = "/content/wiki_json"

PAGES_PER_FILE = 1000
NAMESPACES = "0"
INCLUDE_REDIRECTS = False
LEAD_TITLE = "Introduction"

# Set to None for a full run.
LIMIT = 10
ARCHIVE_LIMIT = 1

In [ ]:
from wikiparser.dump_converter import (
    convert_catalog,
    convert_dump,
    is_catalog_source,
    parse_namespaces,
)

common_kwargs = dict(
    output_dir=OUTPUT_DIR,
    pages_per_file=PAGES_PER_FILE,
    lead_title=LEAD_TITLE,
    namespaces=parse_namespaces(NAMESPACES),
    skip_redirects=not INCLUDE_REDIRECTS,
    limit=LIMIT,
)

if is_catalog_source(SOURCE):
    summary = convert_catalog(
        catalog_url=SOURCE,
        archive_limit=ARCHIVE_LIMIT,
        **common_kwargs,
    )
    print(
        f"Wrote {summary.pages_written} pages from "
        f"{summary.archives_processed}/{summary.archives_seen} archives "
        f"into {summary.files_written} JSON files."
    )
else:
    summary = convert_dump(source=SOURCE, **common_kwargs)
    print(f"Wrote {summary.pages_written} pages into {summary.files_written} JSON files.")

print(f"Output directory: {summary.output_dir}")

In [ ]:
from pathlib import Path
import json

output_path = Path(OUTPUT_DIR)
json_files = sorted(output_path.rglob("pages-*.json"))
print(f"JSON files: {len(json_files)}")

if json_files:
    sample = json.loads(json_files[0].read_text(encoding="utf-8"))
    print(f"First file: {json_files[0]}")
    print(f"Pages in first file: {len(sample)}")
    print(json.dumps(sample[0], ensure_ascii=False, indent=2)[:2000])

In [ ]:
import shutil

zip_path = shutil.make_archive("/content/wiki_json", "zip", OUTPUT_DIR)
print(f"Created archive: {zip_path}")